# Bradley-Terryモデル

**Bradley-Terryモデル**（Bradley & Terry, 1952）は、複数の項目（プレイヤー・製品・案など）を **総当たりのペア比較（pairwise comparison）** にかけたときの勝敗データから、各項目の相対的な「強さ」を推定する統計モデル。

スポーツの対戦成績からチームの強さを推定する、複数のデザイン案を2つずつ比較させてどちらが良いか回答してもらう、といった場面で使われる。前章で見たとおり、これは[選択型コンジョイント分析（CBC）と多項ロジットモデル](choice_based_conjoint.ipynb)における多項ロジットモデルを選択肢数2（ペア比較）に限定した特殊ケースに一致する。

## モデル

$J$個の項目それぞれに強さのパラメータ$\pi_j > 0$（あるいは対数強さ$\beta_j = \log \pi_j$）を割り当てる。項目$i$と項目$j$を比較したとき、$i$が勝つ確率を

$$
P(i \succ j) = \frac{\pi_i}{\pi_i + \pi_j} = \frac{\exp(\beta_i)}{\exp(\beta_i) + \exp(\beta_j)}
$$

と定義する。この式は[選択型コンジョイント分析（CBC）と多項ロジットモデル](choice_based_conjoint.ipynb)で導出した多項ロジットモデルの選択確率

$$
P(y = j \mid \{j,k\}) = \frac{\exp(V_j)}{\exp(V_j) + \exp(V_k)}
$$

において$V_j = \beta_j$としたものと完全に同じ形をしている。パラメータは全体の水準（スケール）が不定なので、通常はどれか1項目の$\beta$を$0$に固定するか、$\sum_j \beta_j = 0$という制約をおいて識別する。

:::{margin} ロジスティック分布との関係
$P(i \succ j) = \dfrac{1}{1+\exp[-(\beta_i - \beta_j)]}$ と変形すると、勝敗確率が2項目の強さの**差**$\beta_i - \beta_j$の標準ロジスティック関数になっていることが分かる。これはコンジョイント分析における2値ロジットモデルと数式上同一であり、Bradley-Terryモデルは「対戦相手を比較対象の属性の1つとみなした2値選択モデル」と解釈できる。
:::

## 推定：最尤法

項目$i$が項目$j$に勝った回数を$w_{ij}$とすると、対数尤度は

$$
\ell(\boldsymbol{\beta}) = \sum_{i < j} \Big[ w_{ij} \log P(i \succ j) + w_{ji} \log P(j \succ i) \Big]
$$

これを最大化する$\boldsymbol{\beta}$は解析的に閉じた解を持たないため、通常はIRLS（反復重み付き最小二乗法）やニュートン法などの数値最適化、あるいは **Zermeloのアルゴリズム（MM algorithm）** と呼ばれる専用の反復更新式で求める。

## 実装例

5つの項目（施策案A〜E）を総当たりでペア比較したデータから、Bradley-Terryモデルで各項目の強さを推定する。

In [2]:
import numpy as np
import pandas as pd
import itertools

rng = np.random.default_rng(0)

items = ["案A", "案B", "案C", "案D", "案E"]
true_beta = {"案A": 1.5, "案B": 0.5, "案C": 0.0, "案D": -0.5, "案E": -1.5}

# 総当たりペアそれぞれについて、n_trials回の比較を行う
n_trials = 20
records = []
for i, j in itertools.combinations(items, 2):
    p_i_wins = np.exp(true_beta[i]) / (np.exp(true_beta[i]) + np.exp(true_beta[j]))
    wins_i = rng.binomial(n_trials, p_i_wins)
    records.append({"winner": i, "loser": j, "count": wins_i})
    records.append({"winner": j, "loser": i, "count": n_trials - wins_i})

df = pd.DataFrame(records)
df.head()


,winner,loser,count
0,案A,案B,14
1,案B,案A,6
2,案A,案C,17
3,案C,案A,3
4,案A,案D,20


In [3]:
def fit_bradley_terry(df, items, n_iter=200):
    # Zermelo(MM algorithm)によるBradley-Terryモデルの推定
    pi = {item: 1.0 for item in items}  # 強さ pi_j = exp(beta_j) を1で初期化

    win_counts = {item: 0 for item in items}
    for _, row in df.iterrows():
        win_counts[row["winner"]] += row["count"]

    for _ in range(n_iter):
        new_pi = {}
        for i in items:
            numerator = win_counts[i]
            denominator = 0.0
            for j in items:
                if j == i:
                    continue
                n_ij = df.query("winner == @i and loser == @j")["count"].sum() \
                     + df.query("winner == @j and loser == @i")["count"].sum()
                denominator += n_ij / (pi[i] + pi[j])
            new_pi[i] = numerator / denominator
        # 識別のため合計を1に正規化
        total = sum(new_pi.values())
        pi = {k: v / total for k, v in new_pi.items()}

    return pi

pi_hat = fit_bradley_terry(df, items)
beta_hat = {k: np.log(v) for k, v in pi_hat.items()}

# 真値と比較するため、案Cを基準(0)にそろえる
offset = beta_hat["案C"]
beta_hat_centered = {k: v - offset for k, v in beta_hat.items()}

pd.DataFrame({"true_beta": true_beta, "estimated_beta": beta_hat_centered})


,true_beta,estimated_beta
案A,1.5,1.845498
案B,0.5,0.347279
案C,0.0,0.000000
案D,-0.5,-0.494119
案E,-1.5,-1.256825


基準を「案C」（真値$0$）にそろえた推定値$\hat\beta$は、真の強さの順序（A > B > C > D > E）とおおよその大きさを正しく復元できている。

## AHPとの違い

Bradley-Terryモデルは各ペアの **勝敗（二値）** または **勝率** という確率的な観測データから強さを推定する統計モデルであるのに対し、[AHP（階層分析法）](ahp.ipynb) は各ペアについて回答者自身が「どちらがどれくらい優れているか」を1〜9段階などで直接評定した値を使い、固有値法で重みを計算する。Bradley-Terryは尤度に基づく統計的推定（標準誤差や信頼区間を計算できる）である一方、AHPは決定論的な行列計算である点が大きな違いである。